In [1]:
import os

from google.protobuf import json_format
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

conversation_history = [
    {
        "role": "system",
        "content": """You are Kopi, a friendly AI assistant for a coffe shop in Malang, Indonesia. You know everything for a coffe, help customers choose drinks, and occasionally share fun facts about Indonesian coffee culture.
        Keep responses warm, short, and conversational."""
    }
]

def chat(user_message):
    #Add user message to history
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Send entire history to API
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=conversation_history,
        temperature=0.7,
        max_tokens=100,
        frequency_penalty=0.3
    )

    # Get API response
    ai_response = response.choices[0].message.content

    #Add AI response to history too
    conversation_history.append({
        "role": "assistant",
        "content": ai_response
    })

    print(f"Input tokens:", response.usage.prompt_tokens)
    print(f"Output tokens:", response.usage.completion_tokens)
    print(f"Total tokens:", response.usage.total_tokens)

    return ai_response

#Main chat loop
print("Kopi Coffee Assistant (type 'quit' to exit)")
print("=" * 40)

while True:
    user_input = input("You: ")

    if user_input.lower() == 'quit':
        print("GoodBye!")
        break

    response = chat(user_input)
    print(f"Kopi: {response}\n")


Kopi Coffee Assistant (type 'quit' to exit)
Input tokens: 90
Output tokens: 24
Total tokens: 114
Kopi: Halo! Welcome to our cozy coffe shop in Malang. What can I get started for you today?

GoodBye!


In [33]:
import os
from groq import Groq
from dotenv import load_dotenv
import json

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": """You are a travel tips assistant.
            Always respond in valid JSON format only.
            No extra text, no explanation, no markdown backticks.
            Just pure JSON."""
        },
        {
            "role": "user",
            "content": "Give me 3 travel tips for Japan"
        }
    ],
    temperature=0  # always use 0 for structured outputs
)

raw = response.choices[0].message.content
data = json.loads(raw)  # convert JSON string → Python dict

print(data["travelTips"][0]["description"])  # clean access!

Learn basic Japanese phrases, such as 'konnichiwa' (hello), 'arigatou' (thank you), and 'sumimasen' (excuse me), to show respect and appreciation for the culture


In [49]:
import os
from groq import Groq
from dotenv import load_dotenv
import json

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

system_prompt = """You are a travel tips assistant.

Always respond with JSON in exactly this structure:
{
    "country": "country name",
    "tips": [
        {
            "title": "short tip title",
            "description": "one sentence explanation"
        }
    ],
    "best_month_to_visit": "month name",
    "budget_per_day_usd": number
}
Return only valid JSON. No extra text."""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": "Give me 3 travel tips for Indonesia"
        }
    ],
    temperature=0   # always use 0 for structured outputs
)
raw = response.choices[0].message.content

raw = raw.replace("```json", "").replace("```", "").strip()

try:
    data = json.loads(raw)                  # convert JSON string → Python dict
    print(data["tips"][0]["description"])
except json.JSONDecodeError:
    print("AI returned invalid JSON:", raw)
    data = None

Indonesia is a predominantly Muslim country, so it's essential to dress modestly and respect local customs, especially when visiting temples or mosques.


In [32]:
import requests
import os
from groq import Groq
from dotenv import load_dotenv
import json

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def get_country_data(country_name):
    try:
        response = requests.get(
            f"https://restcountries.com/v3.1/name/{country_name}", timeout=5
        )
        response.raise_for_status()

        country = response.json()[0]

        return {
            "name": country.get("name", {}).get("common", "Unknown"),
            "capital": country.get("capital", ["Unknown"])[0],
            "population": country.get("population", "Unknown"),
            "region": country.get("region", "Unknown"),
            "currency": list(country.get("currencies", {}).keys())[0] if country.get("currencies") else "Unknown",
            "latitude": country.get("latlng", [0])[0],
            "longitude": country.get("latlng", [0, 0])[1]
        }

    except requests.exceptions.ConnectionError:
        print("No Internet Connection")
        return None
    except requests.exceptions.Timeout:
        print("Request timed out")
        return None
    except requests.exceptions.HTTPError:
        print(f"HTTP error: '{country_name}' not found")
        return None

# FUNCTION 2 - Get weather data

def get_weather(latitude, longitude):
    try:
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "current_weather": True,
            "timezone": "auto"
        }

        response = requests.get(
            "https://api.open-meteo.com/v1/forecast", params=params, timeout=5
        )
        response.raise_for_status()

        weather = response.json().get("current_weather", {})

        return {
            "temperature": weather.get("temperature", "Unknown"),
            "windspeed": weather.get("windspeed", "Unknown")
        }

    except requests.exceptions.RequestException as e:
        print("Weather fetch failed", e)
        return None

def generate_travel_summary(country_data, weather_data):

    system_prompt = """You are a travel data assistant.
    Always respond with JSON in exactly this structure:
    {
        "tagline": "one exciting sentence about the country",
        "unique_facts": ["fact 1", "fact 2"],
        "weather_summary": "one sentence about current weather",
        "tip": "one practical travel tip",
        "rating": a number from 1-10 for how exciting this destination is
    }
    Return only valid JSON. No extra text."""

    user_message = f"""Generate travel summary for {country_data['name']}.
    Temperature: {weather_data['temperature']}°C
    Wind: {weather_data['windspeed']} km/h
    Region: {country_data['region']}
    Population: {country_data['population']}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0,
        max_tokens=400
    )

    raw = response.choices[0].message.content
    raw = raw.replace("```json", "").replace("```", "").strip()
    return raw


def main():
    country_data = get_country_data("indonesia")
    if not country_data:
        return

    weather_data = get_weather(
        country_data["latitude"],
        country_data["longitude"])
    if not weather_data:
        return

    raw = generate_travel_summary(country_data, weather_data)

    try:
        data = json.loads(raw)

        # Now access each field cleanly
        print(f"✨ {data['tagline']}")
        print(f"\n📌 Unique facts:")
        for fact in data['unique_facts']:
            print(f"   - {fact}")
        print(f"\n🌤️  Weather: {data['weather_summary']}")
        print(f"💡 Tip: {data['tip']}")
        print(f"⭐ Excitement rating: {data['rating']}/10")

    except json.JSONDecodeError:
        print("Could not parse response:", raw)

if __name__ == "__main__":
    main()

✨ Indonesia is a vibrant archipelago with over 17,000 islands to explore, offering a rich cultural heritage and breathtaking natural beauty

📌 Unique facts:
   - Indonesia is home to more than 130 active volcanoes
   - The country has over 300 ethnic groups, making it one of the most culturally diverse nations in the world

🌤️  Weather: The current weather in Indonesia is warm with a temperature of 28.2°C and a gentle wind of 7.3 km/h
💡 Tip: Be sure to try the local cuisine, including popular dishes like nasi goreng and gado-gado, and don't forget to respect the local customs and traditions
⭐ Excitement rating: 8/10
